In [ ]:
# ── Standard numerical / plotting imports ──────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# mumax3PP: post-processing helpers for Mumax3 micromagnetic simulations
import mumax3PP.ovf as ovf                          # Read OVF / NPZ field files output by Mumax3
import mumax3PP.parameters as parameters            # Parse simulation parameters / OVF headers
import mumax3PP.fft_across_xyzm as FFT_across_xyzm # FFT helpers along spatial / magnetisation axes
# import beam_processing as bproc                  # (disabled) alternative beam-profile utilities

import matplotlib.colors as colors
from matplotlib import mlab, cm

from scipy.optimize import curve_fit    # Non-linear least-squares fitting
from scipy.signal import argrelextrema # Local extrema detection (unused here but available)

import glob
from os import path
import time

%matplotlib inline

# ── Convenience aliases ────────────────────────────────────────────────────────
sqrt = np.sqrt
pi   = np.pi
exp  = np.exp
sin  = np.sin
cos  = np.cos
mu0  = np.pi * 4e-07   # Vacuum permeability μ₀ (H/m)

plt.rcParams['figure.figsize'] = (12.0, 5.0)  # Default figure size for all plots

# Colour cycle used for multi-series plots (matplotlib default tab10 palette)
pcolors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
]


In [ ]:
# ── Utility and fitting functions ──────────────────────────────────────────────

def findNearest(array, value):
    """Return the index and value of the element in *array* closest to *value*."""
    idx = (np.abs(array - value)).argmin()
    return idx, array[idx]


def fun_lin(x, a, b):
    """Linear model: f(x) = a*x + b.
    Used to fit the linear trajectory of each spin-wave beam segment
    (incident, reflected, split, confluent) after extracting beam-centre positions.
    The slope *a* encodes the propagation angle via θ = arctan(a).
    """
    return a * x + b


def fun_gaus(x, A, x0, w, C):
    """Single Gaussian model: f(x) = A * exp(-(x-x0)²/(2w²)) + C.
    Fitted to each transverse cut-line of the spin-wave beam intensity profile
    to locate the beam centre position x0 at a given y (or x) slice.
    Parameters:
        A  – peak amplitude
        x0 – beam centre position (m)
        w  – beam waist / standard deviation (m)
        C  – constant background offset
    """
    return A * exp(-((x - x0) ** 2) / (2 * w ** 2)) + C


def fun_gaus_2(x, A1, x01, w1, A2, x02, w2, C):
    """Double Gaussian model: sum of two Gaussians + constant background.
    Available for cases where two spatially overlapping beams appear in the
    same cut-line (e.g. incident + reflected beams in the same y-window).
    Parameters mirror fun_gaus for each of the two peaks.
    """
    return (
        A1 * exp(-((x - x01) ** 2) / (2 * w1 ** 2))
        + A2 * exp(-((x - x02) ** 2) / (2 * w2 ** 2))
        + C
    )


In [ ]:
# ── Global simulation / display parameters ─────────────────────────────────────
f_SWB = 45e09   # Carrier frequency of the incident spin-wave beam (Hz); 45 GHz
font  = 12      # Font size for plot annotations


In [ ]:
# ── Output directory for saved figures ────────────────────────────────────────
path_fig = r"/home/users/inelastic_scattering/k_pos/figs"


In [ ]:
# ── Analysis control flags and loop parameters ────────────────────────────────
save_to_file = True   # Write extracted beam angles and GH shifts to a .txt file
save_figs    = False  # Save trajectory overlay figures to disk
show_fits    = False  # Show Gaussian fit diagnostics for every 50th slice

angle = 35            # Beam incidence angle onto the Py edge (degrees)
amp   = "0.0300"      # Edge-mode excitation amplitude label (used in filenames)

# Array of edge-mode frequencies ν to sweep over (GHz)
ff = np.array([11.0,])
ff = np.arange(11.0, 15.51, 0.5, dtype=float)   # Full sweep 11.0–15.5 GHz in 0.5 GHz steps
# ff = np.array(["11.00", "11.50", "12.00", "12.50"])  # (disabled) string-format alternative

# Damping length: number of damping cells × 2 sides × cell size (m)
# Used as a spatial reference / cut-off for near-boundary regions
L_damp = 60 * 2 * 5e-9


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MAIN ANALYSIS LOOP
# For each edge-mode frequency ν in ff this loop:
#   1. Loads the frequency-domain magnetisation M(f, z, y, x) from an NPZ file.
#   2. Extracts the spin-wave beam (SWB) field at the carrier frequency f_SWB
#      and at the edge-mode sideband frequency f_eSW.
#   3. Fits Gaussian profiles to transverse slices of the SWB intensity to
#      locate the beam-centre trajectory at each y (or x) position.
#   4. Fits a straight line through the beam-centre loci to obtain the
#      propagation angle for: incident beam, specularly reflected beam,
#      frequency-downshifted (splitting) beam, and frequency-upshifted
#      (confluence) beam.
#   5. Computes the Goos-Hänchen (GH) lateral shift for each scattered beam
#      as the lateral offset between its trajectory and the incident trajectory
#      at the edge (x = x_max).
#   6. Measures the amplitude of each beam at a fixed x cross-section to
#      derive inelastic scattering efficiencies.
#   7. Writes all results to a tab-delimited text file.
# ══════════════════════════════════════════════════════════════════════════════

# ── Initialise GH-shift accumulators ──────────────────────────────────────────
GH_ref_Li       = []   # GH shift of the specularly reflected beam (nm)
GH_ref_minus_Li = []   # GH shift of the frequency-downshifted (splitting) beam (nm)
GH_ref_plus_Li  = []   # GH shift of the frequency-upshifted (confluence) beam (nm)

# ── Open output text file ─────────────────────────────────────────────────────
if save_to_file:
    file1 = open(
        r'D:\mumax3\inelastic_scattering\k_pos\{}\filtered_amp_{}_ang_{}_deg_k_pos_111.txt'.format(
            angle, amp, angle
        ), 'w+'
    )
    # Header row: all extracted quantities separated by tabs
    file1.write(
        'f (GHz)    inc_ang (deg)    refl_ang (deg)    ref_f_minus_ang (deg)    '
        'ref_f_plus_ang (deg)    GH_ref (nm)    GH_ref_minus (nm)     '
        'GH_ref_plus (nm)    amp_inc    amp_refl    amp_split    amp_conf\n'
    )


# ══════════════════════════════════════════════════════════════════════════════
for fi in ff:   # Iterate over edge-mode frequencies
# ══════════════════════════════════════════════════════════════════════════════

    f_eSW = float(fi) * 1e9   # Edge-mode frequency in Hz

    # ── Load frequency-domain magnetisation data ───────────────────────────────
    # The NPZ file contains M(f, z, y, x, component) — the DFT of m_z over time,
    # pre-computed and stored to avoid re-running the full FFT each time.
    path0 = (
        r"D:\mumax3\inelastic_scattering\k_pos\{}\{}\"
        r"B_300mT_beam_f_45GHz_angle_{}_deg_edge_f_{}GHz_amp_{}_Mz_fzyxm"
        .format(angle, amp, angle, fi, amp)
    )
    print("Data loading...")
    parms    = parameters.ovfParms()
    M_fzyxm  = ovf.OvfFile(path0 + f".npz")
    M_fzyxm.array /= (M_fzyxm.array).shape[0]   # Normalise by number of time steps (DFT normalisation)
    M    = M_fzyxm.array   # Shape: [freq, z_layer, y_cells, x_cells, component]
    fqs  = M_fzyxm.time    # Frequency axis (Hz) corresponding to DFT bins

    # Spatial cell sizes and coordinate arrays (metres)
    cy = (M_fzyxm._headers)['ystepsize']
    cx = (M_fzyxm._headers)['xstepsize']
    y  = np.arange(0, np.shape(M_fzyxm.array)[2], 1) * cy
    x  = np.arange(0, np.shape(M_fzyxm.array)[3], 1) * cx

    # Find frequency indices for the carrier (f_SWB) and edge-mode sideband (f_eSW)
    f_eSW_id      = findNearest(fqs, f_eSW)[0]
    f_eSW_2nd_id  = findNearest(fqs, 2 * f_eSW)[0]   # Second harmonic (not used here)
    f_SWB_id      = findNearest(fqs, f_SWB)[0]

    # 2-D spatial maps at the relevant frequencies (z-layer 0, all y/x)
    M_SWB = M[f_SWB_id, 0, :, :]   # Carrier beam field map
    M_eSW = M[f_eSW_id, 0, :, :]   # Edge-mode sideband field map


    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 1: Incident and specularly reflected beam trajectories
    # Strategy: for each y-slice in a chosen window, fit a Gaussian to the
    # transverse x-profile of |M_SWB| to find the beam-centre x position.
    # Then fit a straight line through (x_centre, y) pairs → beam angle.
    # ══════════════════════════════════════════════════════════════════════════

    # Beam-centre position lists for incident and reflected trajectories
    x_inc, y_inc = [], []
    x_ref, y_ref = [], []
    x_ref_minus, y_ref_minus = [], []   # (reserved for future use)
    x_ref_plus,  y_ref_plus  = [], []   # (reserved for future use)

    # ── y-window for the INCIDENT beam ────────────────────────────────────────
    # Window selected in the lower half of the simulation box where the
    # incident beam propagates before reaching the Py edge.
    if angle in (30, 32.5):
        y0_inc, y1_inc = 400, 600
    elif angle == 35:
        y0_inc, y1_inc = 400, 600
    elif angle == 40:
        y0_inc, y1_inc = 600, 800
    elif angle == 45:
        y0_inc, y1_inc = 600, 800

    nn     = 1   # Step between successive y-slices (1 = every cell)
    slices = np.arange(y0_inc, y1_inc + nn, nn, dtype=int)

    # Quick overview: show the full |M_SWB| map with the y-window highlighted
    plt.figure(figsize=(8, 8))
    plt.imshow(np.abs(M_SWB), aspect='equal', origin="lower",
               extent=[0, x.max(), 0, y.max()])
    plt.axhline(y=y[y0_inc], c='r', ls='--')
    plt.axhline(y=y[y1_inc], c='r', ls='--')
    plt.show()

    for i in slices:
        # Transverse profile of the incident beam at y-row i
        cutline_inc = np.abs(M_SWB[i, :, 0])

        # Fit a Gaussian to locate the beam centre in x
        popt, pcov = curve_fit(
            fun_gaus, x, cutline_inc,
            p0=(cutline_inc.max(),
                x[findNearest(cutline_inc, cutline_inc.max())[0]],
                0.5e-6, 0)
        )

        # Optional: display the fit for every 50th slice (diagnostic)
        if show_fits and i % 50 == 0:
            plt.title(
                r"slice: {} nm, $\nu$={} GHz, BEAM Incident, f = {} GHz".format(
                    round(y[i] * 1e9, 1), fi, round((f_SWB - f_eSW) / 1e9, 1)
                )
            )
            plt.plot(x, cutline_inc, label="incident")
            plt.plot(x, fun_gaus(x, *popt), ls="--", label="FIT")
            plt.legend()
            plt.show()

        x_inc.append(popt[1])   # Gaussian centre x (beam position at this y)
        y_inc.append(y[i])


    # ── y-window for the REFLECTED beam ───────────────────────────────────────
    # Window selected in the upper half of the simulation box where the
    # specularly reflected beam propagates away from the Py edge.
    if angle in (30, 32.5):
        y0_inc, y1_inc = 1200, 1400
    elif angle == 35:
        y0_inc, y1_inc = 1400, 1600
    elif angle == 40:
        y0_inc, y1_inc = 1600, 1800
    elif angle == 45:
        y0_inc, y1_inc = 2000, 2200

    nn     = 1
    slices = np.arange(y0_inc, y1_inc + nn, nn, dtype=int)

    plt.figure(figsize=(8, 8))
    plt.imshow(np.abs(M_SWB), aspect='equal', origin="lower",
               extent=[0, x.max(), 0, y.max()])
    plt.axhline(y=y[y0_inc], c='r', ls='--')
    plt.axhline(y=y[y1_inc], c='r', ls='--')
    plt.show()

    for i in slices:
        cutline_refl = np.abs(M_SWB[i, :, 0])

        popt, pcov = curve_fit(
            fun_gaus, x, cutline_refl,
            p0=(cutline_refl.max(),
                x[findNearest(cutline_refl, cutline_refl.max())[0]],
                0.5e-6, 0)
        )

        if show_fits and i % 50 == 0:
            plt.title(
                r"slice: {} nm, $\nu$={} GHz, BEAM Reflected, f = {} GHz".format(
                    round(y[i] * 1e9, 1), fi, round((f_SWB - f_eSW) / 1e9, 1)
                )
            )
            plt.plot(x, cutline_refl, label="incident")
            plt.plot(x, fun_gaus(x, *popt), ls="--", label="FIT")
            plt.legend()
            plt.show()

        x_ref.append(popt[1])
        y_ref.append(y[i])


    # ── Linear regression of beam-centre loci → propagation angles ───────────
    x_inc = np.array(x_inc);  y_inc = np.array(y_inc)
    x_ref = np.array(x_ref);  y_ref = np.array(y_ref)

    # Fit y(x) = a*x + b for each trajectory; slope a = cot(θ), θ = arctan(a)
    popt_inc, _ = curve_fit(fun_lin, x_inc, y_inc)
    popt_ref, _ = curve_fit(fun_lin, x_ref, y_ref)

    inc   = fun_lin(x, *popt_inc)   # Incident trajectory extrapolated over all x
    ref_f = fun_lin(x, *popt_ref)   # Reflected trajectory extrapolated over all x

    # Propagation angles measured from the x-axis (degrees)
    inc_ang = np.arctan(popt_inc[0]) * (180 / np.pi)
    ref_ang = np.arctan(popt_ref[0]) * (180 / np.pi)

    # ── Overlay trajectory fits on the beam map ───────────────────────────────
    colormap = M_SWB
    plt.figure(figsize=(8, 8))
    plt.title(r"$\nu$={} GHz, f={} GHz".format(f_eSW / 1e9, f_SWB / 1e9))
    plt.imshow(np.abs(colormap), origin="lower", aspect='auto',
               extent=[0, x.max(), 0, y.max()])
    plt.plot(x, inc,   c='red', ls="--", label="incidence, f=50GHz")
    plt.plot(x, ref_f, c='red', ls="--", label="reflected, f=50GHz")
    plt.scatter(x_inc, y_inc, s=2, c='red', marker='X')
    plt.scatter(x_ref, y_ref, s=2, c='red', marker='X')
    plt.xlabel("x"); plt.ylabel("y")
    plt.ylim(0, y.max())
    if save_figs:
        plt.savefig(
            f"{path_fig}/{angle}_deg/incident_reflection_nu_{fi}_GHz_beam_f_{f_SWB/1e9}_GHz.png",
            bbox_inches='tight', pad_inches=0
        )
    plt.show()


    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 2: Frequency-downshifted (splitting) beam trajectory
    # The splitting process produces a beam at f_SWB - f_eSW (inelastic
    # scattering with emission of an edge magnon). Its field map is stored in
    # a separate pre-filtered NPZ file (_split.npz).
    # ══════════════════════════════════════════════════════════════════════════

    dir0  = r'D:\mumax3\inelastic_scattering\k_pos\{}\filtered\\'.format(angle)
    path0 = dir0 + "B_300mT_beam_f_45GHz_angle_{}_deg_edge_f_{}GHz_amp_{}_split.npz".format(angle, fi, amp)
    print("Data loading...")
    npzfile     = np.load(path0)
    x, y, M_SWB_minus = npzfile["x"], npzfile["y"], npzfile["wave"]
    # M_SWB_minus: 2-D spatial field map of the downshifted beam |M(f_SWB - f_eSW)|

    # ── y-window for the SPLITTING beam ───────────────────────────────────────
    # The splitting beam travels at a different angle; window is shifted
    # relative to the reflected-beam window, chosen per incidence angle.
    if angle == 30 and float(fi) < 12.25:
        y0_inc, y1_inc = 1400, 1600
    if angle == 30 and float(fi) > 12.25:
        y0_inc, y1_inc = 1100, 1300
    elif angle == 35:
        y0_inc, y1_inc = 1400, 1600
    elif angle == 40:
        y0_inc, y1_inc = 1600, 1800
    elif angle == 45:
        y0_inc, y1_inc = 1800, 2000

    nn     = 1
    slices = np.arange(y0_inc, y1_inc + nn, nn, dtype=int)

    plt.figure(figsize=(8, 8))
    plt.imshow(np.abs(M_SWB_minus), aspect='equal', origin="lower",
               extent=[0, x.max(), 0, y.max()])
    plt.axhline(y=y[y0_inc], c='r', ls='--')
    plt.axhline(y=y[y1_inc], c='r', ls='--')
    plt.show()

    x_SSP, y_SSP = [], []   # Beam-centre positions for the splitting beam

    for i in slices:
        cutline_SSP = np.abs(M_SWB_minus[i, :])

        # x[:-1] avoids the last cell which may contain a boundary artefact
        popt, pcov = curve_fit(
            fun_gaus, x[:-1], cutline_SSP,
            p0=(cutline_SSP.max(),
                x[findNearest(cutline_SSP, cutline_SSP.max())[0]],
                0.5e-6, 0)
        )

        if show_fits and i % 50 == 0:
            plt.title(
                r"slice: {} nm, $\nu$={} GHz, BEAM SPLITTING, f = {} GHz".format(
                    round(y[i] * 1e9, 1), fi, round((f_SWB - f_eSW) / 1e9, 1)
                )
            )
            plt.plot(x[:-1], cutline_SSP, label="incident")
            plt.plot(x, fun_gaus(x, *popt), ls="--", label="FIT")
            plt.legend()
            plt.show()

        x_SSP.append(popt[1])
        y_SSP.append(y[i])

    x_SSP = np.array(x_SSP);  y_SSP = np.array(y_SSP)
    popt_SSP, _ = curve_fit(fun_lin, x_SSP, y_SSP)
    SSP     = fun_lin(x, *popt_SSP)
    ang_SSP = np.arctan(popt_SSP[0]) * (180 / np.pi)   # Splitting beam angle (deg)

    plt.figure(figsize=(8, 8))
    plt.title(r"SPLITTING, amp {}, $\nu$={} GHz, f-$\nu$={} GHz".format(
        amp, f_eSW / 1e9, (f_SWB - f_eSW) / 1e9))
    plt.imshow(np.abs(M_SWB_minus), origin="lower", aspect='auto',
               extent=[0, x.max(), 0, y.max()])
    plt.plot(x, SSP, c='red', ls="--",
             label=f"reflected, f={round((f_SWB - f_eSW) / 1e9, 1)}GHz")
    plt.scatter(x_SSP, y_SSP, s=2, c='red', marker='X')
    plt.xlabel("x"); plt.ylabel("y")
    plt.ylim(0, y.max())
    if save_figs:
        plt.savefig(
            r"D:\mumax3\inelastic_scattering\k_pos\figs\{}_deg\trajectory_split_amp_{}_nu_{}GHz.png".format(
                angle, amp, fi),
            bbox_inches='tight', pad_inches=0, dpi=300, facecolor="white"
        )
    plt.show()


    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 3: Frequency-upshifted (confluence) beam trajectory
    # The confluence process produces a beam at f_SWB + f_eSW (inelastic
    # scattering with absorption of an edge magnon). Its field map is in
    # a separate pre-filtered NPZ file (_confl.npz).
    # Unlike the incident/reflected/splitting beams (tracked vs y-slices),
    # the confluence beam propagates more steeply and is tracked vs x-slices.
    # ══════════════════════════════════════════════════════════════════════════

    path0 = dir0 + "B_300mT_beam_f_45GHz_angle_{}_deg_edge_f_{}GHz_amp_{}_confl.npz".format(angle, fi, amp)
    print("Data loading...")
    npzfile    = np.load(path0)
    x, y, M_SWB_plus = npzfile["x"], npzfile["y"], npzfile["wave"]
    # M_SWB_plus: 2-D spatial field map of the upshifted beam |M(f_SWB + f_eSW)|

    # ── x-window for the CONFLUENCE beam ──────────────────────────────────────
    # Slices are taken along x (fixed-x cut, varying y) because the confluence
    # beam is nearly parallel to x in some configurations.
    x0_inc, x1_inc = 150, 350

    nn     = 1
    slices = np.arange(x0_inc, x1_inc + nn, nn, dtype=int)

    plt.figure(figsize=(8, 8))
    plt.imshow(np.abs(M_SWB_plus), aspect='equal', origin="lower",
               extent=[0, x.max(), 0, y.max()])
    plt.axvline(x=x[x0_inc], c='r', ls='--')
    plt.axvline(x=x[x1_inc], c='r', ls='--')
    plt.show()

    x_CP, y_CP = [], []   # Beam-centre positions for the confluence beam

    for i in slices:
        # Longitudinal profile at fixed x-column i (y is the spatial coordinate here)
        cutline_CP = np.abs(M_SWB_plus[:, i])

        popt, pcov = curve_fit(
            fun_gaus, y[:-1], cutline_CP,
            p0=(cutline_CP.max(),
                y[findNearest(cutline_CP, cutline_CP.max())[0]],
                0.5e-6, 0)
        )

        if show_fits and i % 50 == 0:
            plt.title(
                r"slice: {} nm, $\nu$={} GHz, BEAM Confluence, f = {} GHz".format(
                    round(y[i] * 1e9, 1), fi, round((f_SWB - f_eSW) / 1e9, 1)
                )
            )
            plt.plot(y[:-1], cutline_CP, label="incident")
            plt.plot(y, fun_gaus(y, *popt), ls="--", label="FIT")
            plt.legend()
            plt.show()

        x_CP.append(x[i])     # x is the independent variable (slice position)
        y_CP.append(popt[1])  # Gaussian centre y = beam position at this x

    x_CP = np.array(x_CP);  y_CP = np.array(y_CP)
    popt_CP, _ = curve_fit(fun_lin, x_CP, y_CP)
    CP     = fun_lin(x, *popt_CP)
    ang_CP = np.arctan(popt_CP[0]) * (180 / np.pi)   # Confluence beam angle (deg)

    plt.figure(figsize=(8, 8))
    plt.title(r"CONFLUENCE, amp {}, $\nu$={} GHz, f+$\nu$={} GHz".format(
        amp, f_eSW / 1e9, (f_SWB + f_eSW) / 1e9))
    plt.imshow(np.abs(M_SWB_plus), origin="lower", aspect='auto',
               extent=[0, x.max(), 0, y.max()])
    plt.plot(x, CP, c='red', ls="--",
             label=f"reflected, f={round((f_SWB - f_eSW) / 1e9, 1)}GHz")
    plt.scatter(x_CP, y_CP, s=2, c='red', marker='X')
    plt.xlabel("x"); plt.ylabel("y")
    plt.ylim(0, y.max())
    if save_figs:
        plt.savefig(
            r"D:\mumax3\inelastic_scattering\k_pos\figs\{}_deg\trajectory_split_amp_{}_nu_{}GHz.png".format(
                angle, amp, fi),
            bbox_inches='tight', pad_inches=0, dpi=300, facecolor="white"
        )
    plt.show()


    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 4: Goos-Hänchen (GH) lateral shifts
    # The GH shift is defined as the lateral (x) displacement between a
    # scattered beam's extrapolated trajectory and the incident beam's
    # trajectory, both evaluated at the Py edge (x = x_max, i.e. index -1).
    # A positive GH shift means the scattered beam exits the edge further
    # along x than the geometrically expected reflection point.
    # ══════════════════════════════════════════════════════════════════════════

    GH_ref       = (ref_f[-1] - inc[-1]) * 1e9   # Specular reflection GH shift (nm)
    GH_ref_minus = (SSP[-1]   - inc[-1]) * 1e9   # Splitting beam GH shift (nm)
    GH_ref_plus  = (CP[-1]    - inc[-1]) * 1e9   # Confluence beam GH shift (nm)

    GH_ref_Li.append(GH_ref)
    GH_ref_minus_Li.append(GH_ref_minus)
    GH_ref_plus_Li.append(GH_ref_plus)


    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 5: Beam amplitude extraction and inelastic scattering efficiency
    # Peak beam amplitudes are sampled at a fixed x cross-section (dx = 4 µm)
    # for the incident and reflected beams.  For the split and confluent beams
    # the x position is corrected for their different propagation angles so
    # that the cross-section is taken at an equivalent propagation distance.
    # Efficiency = amp_scattered / amp_incident (as a fraction or %).
    # ══════════════════════════════════════════════════════════════════════════

    dx = 4e-6   # Reference cross-section distance from the antenna (m)

    # Incident beam: peak amplitude in the lower half of M_SWB at x = dx
    x0 = findNearest(x, dx)[0]
    print("cutline relfected at x = " + str(x[x0] * 1e6) + " mum")
    amp_inc  = np.amax(np.abs(M_SWB[:M_SWB.shape[0] // 2, x0]))

    # Reflected beam: peak amplitude in the upper half at x = dx
    amp_refl = np.amax(np.abs(M_SWB[M_SWB.shape[0] // 2:, x0]))

    # Splitting beam: x position scaled by cos(ang_SSP)/cos(inc_ang) so that
    # the sampled cross-section corresponds to equal propagation distance along
    # the beam axis as for the incident beam.
    x0 = findNearest(
        x,
        dx * (np.cos(np.abs(ang_SSP) * (np.pi / 180)) / np.cos(np.abs(inc_ang) * (np.pi / 180)))
    )[0]
    print("cutline splitting at x = " + str(x[x0] * 1e6) + " mum")
    amp_split = np.amax(np.abs(M_SWB_minus[:, x0]))

    # Confluence beam: same angle-corrected x position
    x0 = findNearest(
        x,
        dx * (np.cos(np.abs(ang_CP) * (np.pi / 180)) / np.cos(np.abs(inc_ang) * (np.pi / 180)))
    )[0]
    print("cutline confluence at x = " + str(x[x0] * 1e6) + " mum")
    amp_conf = np.amax(np.abs(M_SWB_plus[:, x0]))
    print("  "); print("  "); print("  ")


    # ── Write one result row per frequency to the output file ─────────────────
    if save_to_file:
        file1.write('{}\t'.format(fi))
        file1.write('{}\t'.format(inc_ang))
        file1.write('{}\t'.format(ref_ang))
        file1.write('{}\t'.format(ang_SSP))
        file1.write('{}\t'.format(ang_CP))
        file1.write('{}\t'.format(GH_ref))
        file1.write('{}\t'.format(GH_ref_minus))
        file1.write('{}\t'.format(GH_ref_plus))
        file1.write('{}\t'.format(amp_inc))
        file1.write('{}\t'.format(amp_refl))
        file1.write('{}\t'.format(amp_split))
        file1.write('{}\n'.format(amp_conf))

# ── Close the output file after the loop ──────────────────────────────────────
if save_to_file:
    file1.close()

print(" ")
print("DONE")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY PLOTS: GH shifts, scattering angles, and efficiencies vs ν
# Reads the tab-delimited results file produced by Cell 5 and generates three
# scatter-plot summaries for the chosen incidence angle.
# ══════════════════════════════════════════════════════════════════════════════

angle     = 30         # Incidence angle to visualise
amp       = "0.0300"   # Excitation amplitude label
plt.rcParams.update({'font.size': 12})
size      = 100        # Marker size for scatter plots
pic_height = 5
pic_width  = 10
source    = "k_pos"    # Data subdirectory tag

# ── Load the primary results file ─────────────────────────────────────────────
# Columns (0-indexed):
#  0: f (GHz)  1: inc_ang  2: refl_ang  3: ang_SSP  4: ang_CP
#  5: GH_ref   6: GH_ref_minus  7: GH_ref_plus
#  8: amp_inc  9: amp_refl  10: amp_split  11: amp_conf
dane_path = r'D:\mumax3\inelastic_scattering\{}\{}\filtered_amp_{}_ang_{}_deg_{}.txt'.format(
    source, angle, amp, angle, source
)
dane = np.loadtxt(dane_path, skiprows=1)
print(dane.shape)

# ── Load a secondary results file (variant run labelled _111) for comparison ──
dane_path_i = r'D:\mumax3\inelastic_scattering\k_pos\{}\filtered_amp_0.0300_ang_{}_deg_k_pos_111.txt'.format(
    angle, angle
)
dane_i = np.loadtxt(dane_path_i, skiprows=1)


# ── Plot 1: Goos-Hänchen lateral shifts ΔX vs edge-mode frequency ─────────────
# GH_ref_minus (col 6) – downshifted (splitting) beam; red circles
# GH_ref       (col 5) – specular reflection;         green triangles
# GH_ref_plus  (col 7) – upshifted (confluence) beam; blue squares
# Rows 8 (index 8) are skipped in some series to remove an outlier / artefact.
plt.figure(figsize=(pic_width, pic_height))
plt.scatter(dane[:8, 0],  dane[:8, 6],  s=size, c='red',     marker="o", label=r"$f$-$\nu$")
plt.scatter(dane[9:, 0],  dane[9:, 6],  s=size, c='red',     marker="o", label=r"$f$-$\nu$")
plt.scatter(dane_i[:8, 0], dane_i[:8, 6], s=size, c='magenta', marker="o", label=r"$f$-$\nu$")
plt.scatter(dane_i[9:, 0], dane_i[9:, 6], s=size, c='magenta', marker="o", label=r"$f$-$\nu$")
plt.scatter(dane[:, 0],   dane[:, 5],   s=size, c='green',   marker="^", label="$f$=45.0 GHz")
plt.scatter(dane[:4, 0],  dane[:4, 7],  s=size, c='blue',    marker="s", label=r"$f$+$\nu$")
plt.scatter(dane[6:, 0],  dane[6:, 7],  s=size, c='blue',    marker="s", label=r"$f$+$\nu$")
plt.scatter(dane_i[:, 0], dane_i[:, 7], s=size, c='cyan',    marker="s")
plt.xlabel("edge mode frequency (GHz) ")
plt.ylabel(r"$\Delta \it{X}$ (nm)")
plt.xticks(dane[:, 0], dane[:, 0])
# plt.xlim(10.5, 16.)
# plt.legend(fontsize=font, frameon=True)
# plt.savefig(...)
plt.show()


# ── Plot 2: Scattering angles vs edge-mode frequency ─────────────────────────
# Negated to display angles as positive values measured from the surface normal.
# inc_ang (col 1) and refl_ang (col 2) should be equal for specular reflection;
# ang_SSP (col 3) and ang_CP (col 4) differ from inc_ang due to the frequency shift.
plt.figure(figsize=(pic_width, pic_height))
plt.scatter(dane[:8, 0],  -dane[:8, 3],  s=size, c='red',   marker="o", label=r"$f$-$\nu$")
plt.scatter(dane[9:, 0],  -dane[9:, 3],  s=size, c='red',   marker="o", label=r"$f$-$\nu$")
plt.scatter(dane[:, 0],   -dane[:, 2],   s=size, c='green', marker="^", label="$f$=45.0 GHz")
plt.scatter(dane[:, 0],   -dane[:, 4],   s=size, c='blue',  marker="s", label=r"$f$+$\nu$")
plt.xlabel("edge mode frequency (GHz) ")
plt.ylabel(r"angles ($^{\circ}$)")
plt.xticks(dane[:, 0], dane[:, 0])
plt.xlim(10.5, 16.)
# plt.legend(fontsize=font, frameon=True)
# plt.savefig(...)
plt.show()


# ── Plot 3: Inelastic scattering efficiencies vs edge-mode frequency ──────────
# Efficiency = amp_scattered / amp_incident, expressed as a percentage.
# amp_split (col -2 = col 10) and amp_conf (col -1 = col 11) normalised by
# amp_inc (col -4 = col 8).
plt.figure(figsize=(pic_width, pic_height))
plt.scatter(dane[:, 0], (dane[:, -2] / dane[:, -4]) * 100, s=size,
            c='red',  marker="o", label=r"$f$-$\nu$")
plt.scatter(dane[:, 0], (dane[:, -1] / dane[:, -4]) * 100, s=size,
            c='blue', marker="s", label=r"$f$+$\nu$")
plt.xlabel("edge mode frequency (GHz) ")
plt.ylabel(r"efficiency (%)")
plt.xticks(dane[:, 0], dane[:, 0])
plt.xlim(10.5, 16.)
# plt.legend(fontsize=font, frameon=True)
# plt.savefig(...)
plt.show()
